# 01 · Reproducible Research Question & Analysis Preregistration

**Task 1 proof notebook.** It demonstrates: (1) the locked preregistration, (2) the repository structure and environment lockfile, (3) integrity of the raw data, and (4) the **data-blind pipeline check on synthetic fixtures** with the output contract.

Order of events is proven by git history: the preregistration, tests and lockfile were committed and tagged `prereg-v1` **before** any outcome analysis.

In [1]:
import sys, warnings, pathlib, subprocess, hashlib
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from IPython.display import display, Markdown, Image
from rq.config import *
from rq.data import load_raw, add_calendar
from rq import stats_tests as st
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
PAL = {"Event": "#d95f02", "Holiday": "#7570b3", "Regular": "#1b9e77"}
df = add_calendar(load_raw(RAW))
print(f"rows={len(df)}  window={df.date.min().date()} .. {df.date.max().date()}  seed={SEED}")

rows=30  window=2026-06-01 .. 2026-06-30  seed=20260601


## 1. Research question (from `prereg/PREREGISTRATION.md`)

In [2]:
txt = (ROOT / "prereg" / "PREREGISTRATION.md").read_text()
start = txt.index("## Research question"); end = txt.index("## Hypotheses")
display(Markdown(txt[start:end]))

## Research question

**Primary (falsifiable):** In the 30-day observation window 2026-06-01 to 2026-06-30, is mean daily demand on *marketing-event days* different from mean daily demand on *regular days* (no event, no holiday)?

| Element | Definition |
|---|---|
| Population | Daily demand of the single series in `daily-demand-series.csv` (one product/business unit, unit not otherwise identified). Inference is to this series in this window only. |
| Unit of analysis | One calendar day (n = 30). |
| Exposure / intervention | `marketing_event = 1` (n = 8). |
| Comparator | Regular days: `marketing_event = 0` and `holiday = 0` (n = 18). |
| Outcome | `demand` (daily count, raw scale). |
| Observation window | 2026-06-01 (Mon) through 2026-06-30 (Tue), inclusive, daily frequency. |

**Estimand and its limits.** The estimand is the *difference in mean daily demand between event days and regular days*. Because event days are always Fri/Sat and regular days are always Mon-Thu, this contrast is the sum of (a) any causal marketing effect and (b) any Fri/Sat-vs-Mon-Thu weekday effect. The design cannot separate them. Results will be reported as an **association bundled with day-of-week**, never as a causal marketing lift. Observational, single series, no randomisation.

**Secondary questions** (labelled as secondary in all reports):

- S1. Do mean demands differ across the three day types Event / Holiday / Regular (one-way ANOVA, Tukey HSD)?
- S2. Do day type and calendar week interact (two-way ANOVA)?
- S3. Is the series stationary, seasonal at period 7, and forecastable better than a seasonal-naive baseline?
- S4. Is there unsupervised cluster structure in engineered features of the series?

---



## 2. Git history shows the plan preceded the analysis

In [3]:
def sh(cmd):
    return subprocess.run(cmd, cwd=ROOT, shell=True, capture_output=True, text=True).stdout
print(sh("git log --reverse --format='%h  %aI  %s' --decorate"))
print("tags:", sh("git tag").split())
print("files in prereg-v1 commit:\n" + sh("git show --stat --format= prereg-v1 | head -40"))

cdd2c8d  2026-10-11T05:43:07+00:00  Preregistration v1 + synthetic pipeline check + lockfile (BEFORE outcome analysis)
2cc7cfb  2026-10-11T05:44:34+00:00  Fix Ljung-Box burn-in; log deviations (post-lock)

tags: ['prereg-v1']
files in prereg-v1 commit:
tag prereg-v1
Tagger: Analyst <analyst@example.com>

Preregistration locked before inspecting outcome relationships
 .gitignore                              |   5 +
 .python-version                         |   1 +
 Makefile                                |  10 ++
 README.md                               |   3 +
 data/interim/.gitkeep                   |   0
 data/processed/.gitkeep                 |   0
 data/raw/daily-demand-series.csv        |  31 ++++++
 data/raw/daily-demand-series.csv.sha256 |   1 +
 docs/OUTPUT_CONTRACTS.md                |  23 ++++
 prereg/PREREGISTRATION.md               | 166 ++++++++++++++++++++++++++++
 reports/figures/.gitkeep                |   0
 reports/tables/.gitkeep                 |   0
 requirements.i

## 3. Raw-data integrity (hash recorded in the preregistration)

In [4]:
h = hashlib.sha256(RAW.read_bytes()).hexdigest()
recorded = [l for l in txt.splitlines() if "Raw data SHA-256" in l][0]
print("computed :", h)
print("recorded :", recorded)
assert h in recorded, "raw data changed since preregistration!"
print("OK: raw file is byte-identical to the preregistered file")

computed : 171eb7a471f10bfdb4674d34bfa78428c6118823de728245e24239409a053f89
recorded : **Raw data SHA-256:** `171eb7a471f10bfdb4674d34bfa78428c6118823de728245e24239409a053f89` (`data/raw/daily-demand-series.csv`)
OK: raw file is byte-identical to the preregistered file


## 4. Repository structure (raw / interim / processed / notebooks / reports)

In [5]:
skip = {".git", "__pycache__", ".pytest_cache", ".ipynb_checkpoints"}
def tree(p, prefix=""):
    items = sorted([x for x in p.iterdir() if x.name not in skip], key=lambda x: (x.is_file(), x.name))
    for i, x in enumerate(items):
        print(prefix + ("└── " if i == len(items)-1 else "├── ") + x.name)
        if x.is_dir() and x.name != "figures": tree(x, prefix + ("    " if i == len(items)-1 else "│   "))
print(ROOT.name); tree(ROOT)

demand-prereg-repo
├── data
│   ├── interim
│   │   └── .gitkeep
│   ├── processed
│   │   └── .gitkeep
│   └── raw
│       ├── daily-demand-series.csv
│       └── daily-demand-series.csv.sha256
├── docs
│   └── OUTPUT_CONTRACTS.md
├── notebooks
├── prereg
│   └── PREREGISTRATION.md
├── reports
│   ├── figures
│   └── tables
│       └── .gitkeep
├── scripts
│   ├── build_notebooks.py
│   ├── make_lock.py
│   ├── nb_defs_a.py
│   └── nb_defs_b.py
├── src
│   └── rq
│       ├── __init__.py
│       ├── capstone.py
│       ├── clustering.py
│       ├── config.py
│       ├── data.py
│       ├── forecasting.py
│       ├── pipeline.py
│       ├── stats_tests.py
│       └── synthetic.py
├── tests
│   ├── conftest.py
│   └── test_pipeline_synthetic.py
├── .gitignore
├── .python-version
├── Makefile
├── README.md
├── requirements.in
└── requirements.lock


## 5. Environment lockfile
`requirements.in` lists the direct dependencies; `requirements.lock` pins the full transitive closure (generated by `scripts/make_lock.py`).

In [6]:
lock = (ROOT / "requirements.lock").read_text().splitlines()
print(lock[0], "|", len(lock) - 1, "pinned packages")
core = ["numpy", "pandas", "scipy", "statsmodels", "scikit-learn", "matplotlib", "seaborn", "plotly", "nbformat", "reportlab"]
print("\n".join(l for l in lock if l.split("==")[0] in core))

# python 3.12.3 | 78 pinned packages
matplotlib==3.10.8
nbformat==5.11.1
numpy==2.4.4
pandas==3.0.2
plotly==7.1.0
reportlab==4.4.10
scikit-learn==1.8.0
scipy==1.17.1
seaborn==0.13.2
statsmodels==0.15.0


## 6. Data-blind pipeline check on synthetic fixtures
The pipeline (`rq.pipeline.run_pipeline`) accepts *any* dataframe with the raw schema. It was exercised on **synthetic** data with the same calendar structure (Fri/Sat events, Sunday holidays) and **known planted effects**:

* `planted`: +30 units on event days → pipeline must detect a positive effect.
* `null`: no effect → pipeline should usually *not* reject (nominal false-positive rate ≈ 5%).

In [7]:
from rq.synthetic import make_synthetic
from rq.pipeline import run_pipeline, check_contract
rows = {}
for name, kw in {"planted (+30)": dict(seed=1, event_effect=30.0, sd=4.0),
                 "null (0)": dict(seed=11, event_effect=0.0, sd=5.0)}.items():
    res = run_pipeline(make_synthetic(**kw), fast=True)
    check_contract(res)                    # raises AssertionError on any contract violation
    p = res["primary"]
    rows[name] = dict(diff=p["diff"], welch_p=p["welch_p"], mwu_p=p["mwu_p"], hedges_g=p["hedges_g"],
                      primary_test=p["primary_test"], reject=p["reject_null"],
                      k_best=res["k_best"], d=res["d"], forecast_rows=len(res["forecast"]))
display(pd.DataFrame(rows).T)
print("Output contract satisfied on both synthetic fixtures.")

,diff,welch_p,mwu_p,hedges_g,primary_test,reject,k_best,d,forecast_rows
planted (+30),28.888889,0.0,0.000063,7.712761,mann_whitney_u,True,5,0,30
null (0),-0.069444,0.965016,0.759063,-0.014096,welch_t,False,6,1,30


Output contract satisfied on both synthetic fixtures.


In [8]:
# Calibration: false-positive rate of the primary comparison on 400 synthetic null datasets
rej = []
for s in range(400):
    d = add_calendar(make_synthetic(seed=1000 + s))
    a = d.loc[d.day_type == "Event", "demand"]; b = d.loc[d.day_type == "Regular", "demand"]
    rej.append(stats.ttest_ind(a, b, equal_var=False).pvalue < 0.05)
print(f"null false-positive rate at alpha=0.05: {np.mean(rej):.3f}  (nominal 0.05)")

null false-positive rate at alpha=0.05: 0.055  (nominal 0.05)


## 7. Expected output contracts (`docs/OUTPUT_CONTRACTS.md`)

In [9]:
display(Markdown((ROOT / "docs" / "OUTPUT_CONTRACTS.md").read_text()))

# Output contracts (defined before the real outcome was analysed)

`rq.pipeline.run_pipeline(df)` accepts any dataframe with columns
`date, demand, marketing_event, holiday` (consecutive daily dates, binary flags) and must return
the following. `rq.pipeline.check_contract` and `tests/test_pipeline_synthetic.py` enforce them.

| Artifact | Contract |
|---|---|
| `calendar` | input + `dow` (0-6), `t` (0..n-1), `week` (>=1), `day_type` in {Event, Holiday, Regular, Both} |
| `normality` | one row per Event/Holiday/Regular; columns `group, n, shapiro_W, shapiro_p, ks_D, ks_p, lilliefors_D, lilliefors_p, normal_by_shapiro` |
| `primary` | dict with `n_a, n_b, mean_a, mean_b, diff, welch_t/df/p, welch_ci_low/high, hedges_g (+CI), mwu_U/p, rank_biserial, hl_shift (+CI), perm_p, primary_test, primary_p, reject_null`; all p in [0,1]; Welch CI contains `diff` |
| `oneway`, `tukey` | ANOVA dict (F, p, eta2, omega2, Brown-Forsythe p, Kruskal p); Tukey table with exactly 3 pairs, adjusted p in [0,1] |
| `twoway` | table with rows `C(week)`, `C(day_type)`, `C(week):C(day_type)`, `Residual` |
| `robustness` | permutation p, OLS(+trend) coef/CI/p, log-scale Welch p; `leave_one_week_out` table |
| `pca` | explained-variance ratios sum to 1; cumulative non-decreasing; `n_keep` = smallest k with cumulative >= 0.80 (min 2) |
| `kmeans_table` | k = 2..6, silhouette in [-1, 1], `k_best` = argmax silhouette |
| `dbscan`, `hier` | labels length = rows used; hierarchical uses `k_best` clusters |
| `adf`, `d` | ADF table; integer d in {0,1,2} |
| `sarima_grid` | candidates sorted by AICc ascending, training window only |
| `holdout` | models {SARIMA, SeasonalNaive, Naive}; MAPE, RMSE, MAE >= 0; last 7 days |
| `forecast` | exactly 30 rows, first date = last observed date + 1 day, no NaN, `lo95 < lo80 < forecast < hi80 < hi95` |
| Determinism | two runs with the same data and seed give identical numbers |
| Data-blindness | pipeline never reads `data/raw`; the file is only loaded by notebooks/scripts |


## 8. Full synthetic test-suite (pytest)

In [10]:
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-v", "-W", "ignore", "--no-header", "-p", "no:cacheprovider"],
                   cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-3500:])
assert r.returncode == 0

============================= test session starts ==============================
collecting ... collected 18 items

tests/test_pipeline_synthetic.py::test_schema_rejects_bad_inputs PASSED  [  5%]
tests/test_pipeline_synthetic.py::test_calendar_derivation PASSED        [ 11%]
tests/test_pipeline_synthetic.py::test_full_output_contract PASSED       [ 16%]
tests/test_pipeline_synthetic.py::test_group_sizes_match_prereg PASSED   [ 22%]
tests/test_pipeline_synthetic.py::test_planted_effect_recovered PASSED   [ 27%]
tests/test_pipeline_synthetic.py::test_null_false_positive_rate_nominal PASSED [ 33%]
tests/test_pipeline_synthetic.py::test_normality_and_rule_selection PASSED [ 38%]
tests/test_pipeline_synthetic.py::test_twoway_has_interaction_row PASSED [ 44%]
tests/test_pipeline_synthetic.py::test_pca_contract PASSED               [ 50%]
tests/test_pipeline_synthetic.py::test_kmeans_picks_argmax_silhouette PASSED [ 55%]
tests/test_pipeline_synthetic.py::test_clustering_recovers_two_obvious_b

## Conclusion
The research question, hypotheses, exclusions, transformations, tests, effect sizes and robustness checks are fixed in `prereg/PREREGISTRATION.md`; the environment is pinned; the raw data hash is recorded; and the data-blind synthetic pipeline test passes. Real-outcome analysis starts only in notebook 02.